In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:

import os

TF_DETERMINISTIC_OPS = '1'
TF_CUDNN_DETERMINISTIC = '1'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'  # For CuDNN ops

import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import pickle
import time
import yaml
import tensorflow as tf
from tensorflow import keras

In [ ]:
!pip install git+https://github.com/enricodidoli-source/memory-save.git
!pip install git+https://github.com/enricodidoli-source/seed-manager.git
!pip install --no-cache-dir --force-reinstall git+https://github.com/enricodidoli-source/seed-manager.git

from memory_utils import clear_RAM, MemorySave
from seed_manager import SeedManager


In [ ]:

#fixed_path = '/content/drive/MyDrive/BinCellCNN/'
fixed_path = 'C:/Users/admin/BinCellCNN/'

whole_exp = 'End_to_End_pipeline'

if fixed_path not in sys.path:
    sys.path.append(fixed_path)

modules_dir = f'{fixed_path}src/'
if modules_dir not in sys.path:
    sys.path.append(modules_dir)

    
experiment_path = f'{fixed_path}experiments/experiment_{whole_exp}/'
os.makedirs(experiment_path, exist_ok=True)

config_file_path = os.path.join(experiment_path, "config.yml")
with open(config_file_path, "r") as f:
    config = yaml.safe_load(f)


# --- Data ---
train_data_folder_path = config['train_data_folder_path']
labels_level = config['labels_level']

# --- Seed parameters ---
starting_seed = config['starting_seed']

# --- Environment parameters ---
colab = config['colab']
save_memory = config['save_memory']




if save_memory:
    if colab:
        mem_path = '/content/temp_data'
    else:
        mem_path = f'{fixed_path}/temp_data'

    mem = MemorySave(base_dir = mem_path)
else:
    mem = None



# Single Cell Run

In [ ]:

decache_files = ['src.shared.utils', 'src.shared.timepoints_elaboration', 'src.shared.temp_storage', 'src.shared.cv_folds',
                 'src.sample_level.bayesian_tuning', 'src.sample_level.show_results', 'src.sample_level.training',
                 'src.sample_level.testing', 'src.sample_level.configs' 'src.sample_level.pipelines',
                 'src.BinCellCNN.model']

# Remove from cache
from src.shared.utils import remove_from_cache
remove_from_cache(decache_files)


from src.shared.timepoints_elaboration import (load_data, get_sample_ids, take_first_n_patients,  
                                                get_sample_perc, show_blast_distribution_perc)
from src.shared.temp_storage import save_test_temp
from src.shared.cv_folds import generate_LOPOCV_dicts, generate_LOPOCV_folds
from src.shared.utils import load_element, save_element

from src.sample_level.bayesian_tuning import BO_tuning
from src.sample_level.show_results import show_all_LOPO_boxplots, build_results_df
from src.sample_level.training import fit
from src.sample_level.testing import predict
from src.sample_level.configs import get_eval_metric
from src.sample_level.pipelines import run_lopocv_fold

from src.BinCellCNN.model import BinCellCNN



In [ ]:
%%time

all_datasets, all_datasets_labels, multiple_donations, samples_info_df = load_data(config = config,
                                                                                   data_folder_path = train_data_folder_path,
                                                                                    mem = mem)

In [ ]:
if labels_level == 'cell':
    tot_perc_list = show_blast_distribution_perc(samples_info_df = samples_info_df,
                                                return_perc = True,
                                                decimals = 4,
                                                samples_ids = True)
    print(tot_perc_list)

In [ ]:

multiple_donations = take_first_n_patients( config, multiple_donations)

seed_manager = SeedManager(starting_seed=starting_seed, LOPO_fold_idx=0)


full_LOPOCV_dicts = generate_LOPOCV_dicts(config = config, 
                                          multiple_donations =  multiple_donations,
                                          ALL_DATASETS= all_datasets,
                                          mem = mem)

LOPOCV_patients_folds = generate_LOPOCV_folds(config = config, 
                                              full_LOPOCV_dicts = full_LOPOCV_dicts,
                                              ALL_DATASETS= all_datasets,
                                              mem = mem,
                                              seed_manager = seed_manager)

In [ ]:
%%time



results_save_dir = f'{experiment_path}/results/'
os.makedirs(results_save_dir, exist_ok=True)


for LOPOCV_fold_idx, (train_kfolds, test_pat) in enumerate(LOPOCV_patients_folds): 

    test_samples_preds, original_test_y, best_thr = run_lopocv_fold(BinCellCnn = BinCellCNN,
                                                          config = config,
                                                          experiment_path = experiment_path,
                                                          LOPOCV_fold_idx = LOPOCV_fold_idx,
                                                          train_kfolds = train_kfolds,
                                                          test_pat = test_pat,
                                                          multiple_donations = multiple_donations,
                                                          ALL_DATASETS = all_datasets,
                                                          ALL_DATASETS_LABELS = all_datasets_labels,
                                                          mem = mem)


    # === Prepare data for plot === #

    patient_samples_info_df = samples_info_df[samples_info_df['patient_id'] == float(test_pat[0])]
    samples_ids = get_sample_ids(patient_samples_info_df)
    if labels_level == 'cell':
        percs = get_sample_perc(patient_samples_info_df)
    else:
       percs = None

    fold_box_data = []
    for k, (sample_pred, true_y) in enumerate(zip(np.array(test_samples_preds).T.tolist(),
                                                  original_test_y)):
        print(sample_pred)
        box_data = {"True_Label": true_y,
                    'CV_Fold_Preds': sample_pred,
                    'samples_ids': samples_ids[k],
                    'best_thr': best_thr
                    }

        if percs is not None:
            box_data['perc'] = percs[k]

        fold_box_data.append(box_data)


    save_element(results_save_dir, f'fold_box_data_{LOPOCV_fold_idx}', fold_box_data)



results_df = build_results_df(results_save_dir, len(LOPOCV_patients_folds))
show_all_LOPO_boxplots(results_df, save_dir = results_save_dir, file_name = 'Positivity_Score_boxplots.pdf', title = None, fold_or_seed = 'Fold')

#mem.close()